Cell 1 — Install dependencies

In [ ]:
!pip install transformers datasets torch scikit-learn pandas requests flask pyngrok -q
print("✅ Dependencies terinstall")

Cell 2 — Konfigurasi

In [ ]:
import os, uuid

# ── ISI BAGIAN INI ─────────────────────────────────────────────
BACKEND_URL = "https://lurk-imprint-majestic.ngrok-free.dev/api"  # ganti dengan URL backend kamu
COLAB_API_KEY = "your-colab-api-key-2"   # sama dengan COLAB_API_KEY di .env
NGROK_AUTH_TOKEN = "38kxfK49wnqurRRGTAmNAjebCiJ_5ucQEndZh197Mi7bZCQBo"          # dari https://dashboard.ngrok.com
# ──────────────────────────────────────────────────────────────

COLAB_PORT = 5007
WORK_DIR = "/content/transmotion"
SESSION_ID = str(uuid.uuid4())[:8]  # ID unik per sesi Colab

os.makedirs(WORK_DIR, exist_ok=True)

# Header untuk panggilan ke backend
BACKEND_HEADERS = {
    "X-Colab-Key": COLAB_API_KEY,
    "Content-Type": "application/json",
}

print(f"✅ Konfigurasi selesai")
print(f"   Backend    : {BACKEND_URL}")
print(f"   Session ID : {SESSION_ID}")
print(f"   Work dir   : {WORK_DIR}")

Cell 3 — Helper functions

In [ ]:
import json
import math
import os
import random
import time
import traceback
from collections import Counter

import numpy as np
import pandas as pd
import requests
import torch
from datasets import Dataset as HFDataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

# Tambahkan di Cell 3, setelah import

from google.colab import drive
import shutil

def save_model_to_drive(model, tokenizer, job_id, id2label, label2id):
    """
    Simpan full model ke Google Drive agar bisa dimuat saat inference.
    Return path di Drive.
    """
    # Mount Drive jika belum
    if not os.path.exists("/content/drive/MyDrive"):
        print("📁 Mounting Google Drive...")
        drive.mount("/content/drive", force_remount=False)

    drive_dir = f"/content/drive/MyDrive/transmotion_models/job_{job_id[:8]}"
    os.makedirs(drive_dir, exist_ok=True)

    # Simpan model + tokenizer dalam format safetensors (lebih kecil)
    print(f"💾 Menyimpan model ke Drive: {drive_dir}")
    model.save_pretrained(drive_dir, safe_serialization=True)
    tokenizer.save_pretrained(drive_dir)

    # Simpan label map
    with open(os.path.join(drive_dir, "label_map.json"), "w") as f:
        json.dump({str(i): l for i, l in id2label.items()}, f, ensure_ascii=False, indent=2)

    size_mb = sum(
        os.path.getsize(os.path.join(drive_dir, f))
        for f in os.listdir(drive_dir)
    ) / 1024 / 1024
    print(f"   Ukuran total: {size_mb:.1f} MB")
    print(f"   Path: {drive_dir}")

    return drive_dir


def upload_small_weights(model, job_id):
    """
    Upload HANYA classifier head weights ke backend (sangat kecil, ~10KB).
    Ini untuk metadata — model penuh ada di Drive.
    """
    import io

    # Ambil hanya layer classifier
    classifier_state = {
        k: v for k, v in model.state_dict().items()
        if "classifier" in k or "pooler" in k
    }

    buffer = io.BytesIO()
    torch.save(classifier_state, buffer)
    buffer.seek(0)

    size_kb = buffer.getbuffer().nbytes / 1024
    print(f"   Classifier weights: {size_kb:.1f} KB")

    return buffer


# ── HTTP helpers ───────────────────────────────────────────────

def backend_get(path, params=None):
    try:
        res = requests.get(
            f"{BACKEND_URL}{path}",
            headers={k: v for k, v in BACKEND_HEADERS.items() if k != "Content-Type"},
            params=params,
            timeout=30,
        )
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print(f"❌ GET {path}: {e}")
        return None


def backend_post(path, json_data=None, form_data=None, files=None):
    headers = {k: v for k, v in BACKEND_HEADERS.items() if k != "Content-Type"}
    try:
        if files:
            res = requests.post(
                f"{BACKEND_URL}{path}",
                headers=headers,
                data=form_data or {},
                files=files,
                timeout=120,
            )
        else:
            headers["Content-Type"] = "application/json"
            res = requests.post(
                f"{BACKEND_URL}{path}",
                headers=headers,
                json=json_data or {},
                timeout=30,
            )
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print(f"❌ POST {path}: {e}")
        return None


# ── Reproducibility ────────────────────────────────────────────

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ── Dataset ────────────────────────────────────────────────────

def load_dataset_from_backend(job):
    dataset_id = job["dataset_id"]
    print(f"📦 Mengambil dataset: {job.get('dataset_name', dataset_id)}")

    all_rows, page = [], 1
    while True:
        res = backend_get(
            f"/colab/datasets/{dataset_id}/preprocessed",
            params={"page": page, "per_page": 10000},
        )
        if not res or not res.get("data"):
            print("   Dataset kosong")
            print(res)
            break
        all_rows.extend(res["data"])
        total_pages = res.get("meta", {}).get("pagination", {}).get("total_pages", 1)
        print(f"   Halaman {page}/{total_pages} — {len(all_rows)} baris...", end="\r")
        if page >= total_pages:
            break
        page += 1

    print(f"\n✅ {len(all_rows)} baris diambil")
    if not all_rows:
        raise ValueError("Dataset kosong")

    df = pd.DataFrame(all_rows)[["preprocessed_text", "label"]].dropna()
    df = df[df["preprocessed_text"].str.strip() != ""].reset_index(drop=True)

    print("   Distribusi kelas:")
    for lbl, cnt in df["label"].value_counts().items():
        print(f"   - {lbl}: {cnt} ({cnt/len(df)*100:.1f}%)")

    return df


def stratified_split(df, test_size, seed=42):
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(
        df, test_size=test_size, stratify=df["label"], random_state=seed
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    print(f"\n📊 Split: train={len(train_df)}, test={len(test_df)}")
    return train_df, test_df


def build_label_maps(labels):
    unique = sorted(set(labels))
    label2id = {l: i for i, l in enumerate(unique)}
    id2label = {i: l for l, i in label2id.items()}
    return label2id, id2label


def tokenize(df, tokenizer, label2id, max_length):
    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    hf = HFDataset.from_dict({
        "text": df["preprocessed_text"].tolist(),
        "labels": [label2id[l] for l in df["label"].tolist()],
    })
    return hf.map(tokenize_fn, batched=True, remove_columns=["text"], desc="Tokenisasi")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds, average="weighted", zero_division=0)),
    }

# Cache model yang sudah dimuat agar tidak reload tiap request
_loaded_models = {}

def get_model_for_inference(model_id, file_path, base_model_name, num_labels, label_map):
    """
    Load model dari Google Drive ke memori.
    Di-cache agar tidak reload tiap request.
    """
    if model_id in _loaded_models:
        return _loaded_models[model_id]

    print(f"📥 Memuat model {model_id[:8]} dari Drive...")
    print(f"   Path: {file_path}")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Model tidak ditemukan di Drive: {file_path}")

    tokenizer = AutoTokenizer.from_pretrained(file_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        file_path,
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
    )
    model.eval()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    _loaded_models[model_id] = {
        "tokenizer": tokenizer,
        "model": model,
        "label_map": label_map,
        "device": device,
    }

    print(f"✅ Model dimuat ke {device.upper()}")
    return _loaded_models[model_id]


def run_inference(text, model_id, file_path, base_model_name, num_labels, label_map):
    """Jalankan klasifikasi teks."""
    loaded = get_model_for_inference(
        model_id, file_path, base_model_name, num_labels, label_map
    )

    tokenizer = loaded["tokenizer"]
    model = loaded["model"]
    device = loaded["device"]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]

    lm = label_map if isinstance(label_map, dict) else json.loads(label_map)
    all_scores = {
        lm.get(str(i), str(i)): round(float(p), 4)
        for i, p in enumerate(probs)
    }
    predicted_index = int(probs.argmax())
    predicted_label = lm.get(str(predicted_index), str(predicted_index))
    confidence = round(float(probs[predicted_index]), 4)

    return {
        "predicted_label": predicted_label,
        "predicted_index": predicted_index,
        "confidence": confidence,
        "all_scores": all_scores,
    }


# ── Progress Callback ──────────────────────────────────────────

class BackendProgressCallback(TrainerCallback):
    def __init__(self, job_id, total_epochs):
        self.job_id = job_id
        self.total_epochs = total_epochs
        self._last_reported_epoch = 0

    def _find_metrics(self, log_history, epoch):
        """
        Cari train_loss dan eval metrics dari log_history.
        Train loss ada di entry {'loss': ..., 'epoch': N.0}
        Eval metrics ada di entry {'eval_loss': ..., 'eval_accuracy': ..., 'epoch': N.0}
        """
        train_loss = None
        val_loss = None
        val_accuracy = None
        val_f1 = None

        for entry in reversed(log_history):
            entry_epoch = entry.get("epoch", 0)

            # Cari train loss untuk epoch ini
            if train_loss is None and "loss" in entry:
                if abs(entry_epoch - epoch) < 0.5:
                    train_loss = entry["loss"]

            # Cari eval metrics untuk epoch ini
            if val_loss is None and "eval_loss" in entry:
                if abs(entry_epoch - epoch) < 0.5:
                    val_loss = entry.get("eval_loss")
                    val_accuracy = entry.get("eval_accuracy")
                    val_f1 = entry.get("eval_f1")

            # Stop jika sudah dapat keduanya
            if train_loss is not None and val_loss is not None:
                break

        return train_loss, val_loss, val_accuracy, val_f1

    def on_log(self, args, state, control, logs=None, **kwargs):
        """
        on_log dipanggil setiap kali ada entry baru di log_history,
        termasuk setelah eval selesai — lebih reliable dari on_epoch_end.
        """
        if logs is None:
            return

        # Hanya proses saat ada eval_loss (artinya satu epoch baru saja selesai evaluasi)
        if "eval_loss" not in logs:
            return

        current_epoch = int(state.epoch)

        # Hindari lapor epoch yang sama dua kali
        if current_epoch <= self._last_reported_epoch:
            return
        self._last_reported_epoch = current_epoch

        progress = int((current_epoch / self.total_epochs) * 100)
        train_loss, val_loss, val_accuracy, val_f1 = self._find_metrics(
            state.log_history, current_epoch
        )

        payload = {
            "current_epoch": current_epoch,
            "total_epochs": self.total_epochs,
            "progress": progress,
            "train_loss": round(float(train_loss), 4) if train_loss is not None else None,
            "val_loss": round(float(val_loss), 4) if val_loss is not None else None,
            "val_accuracy": round(float(val_accuracy), 4) if val_accuracy is not None else None,
            "val_f1": round(float(val_f1), 4) if val_f1 is not None else None,
            "colab_session_id": SESSION_ID,
        }

        result = backend_post(
            f"/colab/jobs/{self.job_id}/progress",
            json_data=payload,
        )

        status = "✅" if result else "⚠️"
        print(
            f"\n{status} Epoch {current_epoch}/{self.total_epochs} dilaporkan | "
            f"train_loss={payload['train_loss']} | "
            f"val_loss={payload['val_loss']} | "
            f"val_acc={payload['val_accuracy']} | "
            f"val_f1={payload['val_f1']}"
        )


# ── Training utama ─────────────────────────────────────────────

MODEL_BASE_NAMES = {
    "mbert": "bert-base-multilingual-cased",
    "xlmr": "xlm-roberta-base",
}


def train_model(job):
    set_seed(42)

    job_id = job["id"]
    model_type = job["model_type"]
    hp = job["hyperparams"]
    split_info = job.get("split_info", {})

    epochs = hp.get("epochs", 3)
    batch_size = hp.get("batch_size", 16)
    max_length = hp.get("max_length", 128)
    lr = hp.get("learning_rate", 2e-5)
    warmup_steps = hp.get("warmup_steps", 0)
    weight_decay = hp.get("weight_decay", 0.01)
    test_size = split_info.get("test_size", 0.2)
    base_model = MODEL_BASE_NAMES.get(model_type, "bert-base-multilingual-cased")

    print("=" * 55)
    print(f"🚀 Training Job: {job_id[:8]}")
    print(f"   Model: {base_model}")
    print(f"   Epochs: {epochs} | Batch: {batch_size} | LR: {lr}")
    print("=" * 55)

    # 1. Load dataset
    df = load_dataset_from_backend(job)
    train_df, test_df = stratified_split(df, test_size)
    label2id, id2label = build_label_maps(df["label"])
    print(f"\n🏷️  {len(label2id)} kelas: {list(label2id.keys())}")

    # 2. Load model
    print(f"\n📥 Memuat model: {base_model}")
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"   Device: {device.upper()}")

    # 3. Tokenisasi
    print("\n🔤 Tokenisasi...")
    train_ds = tokenize(train_df, tokenizer, label2id, max_length)
    eval_ds = tokenize(test_df, tokenizer, label2id, max_length)

    # 4. Training
    output_dir = os.path.join(WORK_DIR, f"job_{job_id[:8]}")
    os.makedirs(output_dir, exist_ok=True)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=10,
        save_total_limit=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
        callbacks=[
            BackendProgressCallback(job_id, epochs),
            EarlyStoppingCallback(early_stopping_patience=2),
        ],
    )

    print("\n🏋️  Training dimulai...")
    trainer.train()

    # 5. Evaluasi akhir
    print("\n📊 Evaluasi akhir...")
    pred_output = trainer.predict(eval_ds)
    y_pred = pred_output.predictions.argmax(axis=-1)
    y_true = test_df["label"].map(label2id).values

    accuracy = float(accuracy_score(y_true, y_pred))
    f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    precision = float(precision_score(y_true, y_pred, average="weighted", zero_division=0))
    recall = float(recall_score(y_true, y_pred, average="weighted", zero_division=0))

    print(f"\n   Accuracy  : {accuracy*100:.2f}%")
    print(f"   F1 Score  : {f1*100:.2f}%")
    print(f"   Precision : {precision*100:.2f}%")
    print(f"   Recall    : {recall*100:.2f}%")
    print("\n" + classification_report(y_true, y_pred, target_names=list(id2label.values()), zero_division=0))

    # 6. Simpan model ke Drive + upload metadata ke backend
    print(f"\n💾 Menyimpan model ke Google Drive...")
    drive_path = save_model_to_drive(model, tokenizer, job_id, id2label, label2id)

    # Upload classifier weights (kecil) sebagai file ke backend
    # Model penuh ada di Drive dengan path yang disimpan sebagai metadata
    print("\n📤 Mengupload metadata ke backend...")
    classifier_buffer = upload_small_weights(model, job_id)

    job_name = job.get("job_name") or f"{model_type.upper()} Model"
    model_name = f"{job_name} — {time.strftime('%Y%m%d_%H%M')}"
    label_map_json = json.dumps({str(i): l for i, l in id2label.items()})

    try:
        result = backend_post(
            f"/colab/jobs/{job_id}/complete",
            form_data={
                "model_name": model_name,
                "accuracy": str(accuracy),
                "f1_score": str(f1),
                "precision": str(precision),
                "recall": str(recall),
                "label_map": label_map_json,
                "base_model_name": base_model,
                "colab_session_id": SESSION_ID,
                "file_path": drive_path,  # simpan path Drive sebagai metadata
            },
            files={
                "model_file": (
                    f"classifier_{job_id[:8]}.pt",
                    classifier_buffer,
                    "application/octet-stream",
                )
            },
        )
    except Exception as e:
        print(f"❌ Upload gagal: {e}")
        result = None

    if result and result.get("success"):
        print(f"✅ Berhasil! Model ID: {result['data']['id']}")
        print(f"   Drive path: {drive_path}")
    else:
        print("❌ Upload metadata gagal")
        print(f"   Model masih tersimpan di Drive: {drive_path}")

    # Cleanup WORK_DIR saja, Drive tetap ada
    import shutil
    shutil.rmtree(os.path.join(WORK_DIR, f"job_{job_id[:8]}"), ignore_errors=True)

    return {"accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}


def report_failure(job_id, error_message):
    backend_post(
        f"/colab/jobs/{job_id}/fail",
        json_data={"error_message": str(error_message)[:1000]},
    )


print("✅ Training functions siap")
print(f"   GPU: {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ tidak tersedia'}")

Cell 4 — Flask Server and Ngrok

In [ ]:
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf

# ── Setup ngrok ────────────────────────────────────────────────
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# ── Cache model inference ──────────────────────────────────────
_loaded_models = {}

# ── Flask mini-server ──────────────────────────────────────────
colab_app = Flask(__name__)


# ── Helper auth ────────────────────────────────────────────────
def _check_key():
    key = request.headers.get("X-Backend-Key", "")
    return key == COLAB_API_KEY


# ── Routes ─────────────────────────────────────────────────────

@colab_app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "session_id": SESSION_ID,
        "gpu": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "loaded_models": list(_loaded_models.keys()),
    })


@colab_app.route("/train", methods=["POST"])
def receive_train():
    if not _check_key():
        return jsonify({"error": "Unauthorized"}), 401

    job = request.get_json()
    if not job or not job.get("id"):
        return jsonify({"error": "Body tidak valid"}), 400

    job_id = job["id"]

    # Konfirmasi ke backend
    backend_post(
        f"/colab/jobs/{job_id}/running",
        json_data={"colab_session_id": SESSION_ID},
    )

    # Training di background thread
    def run():
        try:
            train_model(job)
        except Exception as e:
            error_msg = traceback.format_exc()
            print(f"\n❌ Training error: {e}")
            report_failure(job_id, error_msg)

    threading.Thread(target=run, daemon=True).start()
    print(f"\n🔔 Job {job_id[:8]} diterima, training dimulai di background")
    return jsonify({"status": "started", "job_id": job_id, "session_id": SESSION_ID})


@colab_app.route("/predict", methods=["POST"])
def predict():
    if not _check_key():
        return jsonify({"error": "Unauthorized"}), 401

    data = request.get_json()
    if not data:
        return jsonify({"error": "Body tidak valid"}), 400

    missing = [f for f in ["model_id", "file_path", "text"] if not data.get(f)]
    if missing:
        return jsonify({"error": f"Field diperlukan: {', '.join(missing)}"}), 400

    # Mount Drive jika belum
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            print("📁 Google Drive di-mount")
        except Exception as e:
            return jsonify({"error": f"Gagal mount Drive: {e}"}), 500

    try:
        result = run_inference(
            text=data["text"],
            model_id=data["model_id"],
            file_path=data["file_path"],
            base_model_name=data.get("base_model_name"),
            num_labels=data.get("num_labels", 2),
            label_map=data.get("label_map", {}),
        )
        return jsonify({"success": True, "data": result})

    except FileNotFoundError as e:
        return jsonify({"success": False, "error": str(e)}), 404
    except Exception as e:
        return jsonify({
            "success": False,
            "error": str(e),
            "detail": traceback.format_exc(),
        }), 500


@colab_app.route("/models/unload/<model_id>", methods=["POST"])
def unload_model(model_id):
    if not _check_key():
        return jsonify({"error": "Unauthorized"}), 401

    if model_id in _loaded_models:
        del _loaded_models[model_id]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return jsonify({"success": True, "message": f"Model {model_id[:8]} di-unload"})

    return jsonify({"success": False, "message": "Model tidak ada di cache"})


@colab_app.route("/shutdown", methods=["POST"])
def shutdown():
    if not _check_key():
        return jsonify({"error": "Unauthorized"}), 401

    backend_post("/colab/unregister", json_data={"session_id": SESSION_ID})
    return jsonify({"status": "shutting down"})


# ── Verifikasi semua route terdaftar ───────────────────────────
print("📋 Routes yang terdaftar:")
with colab_app.app_context():
    for rule in colab_app.url_map.iter_rules():
        print(f"   {list(rule.methods - {'HEAD', 'OPTIONS'})} {rule.rule}")

# ── Jalankan Flask SETELAH semua route didefinisikan ──────────
flask_thread = threading.Thread(
    target=lambda: colab_app.run(
        port=COLAB_PORT,
        debug=False,
        use_reloader=False,
    ),
    daemon=True,
)
flask_thread.start()
time.sleep(2)
print(f"\n✅ Flask server berjalan di port {COLAB_PORT}")

Cell 5 — ngrok Tunnel + Register ke Backend

In [ ]:
from pyngrok import ngrok

# ── Tutup semua tunnel lama sebelum buat yang baru ─────────────
print("🔄 Menutup tunnel lama...")
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
        print(f"   Closed: {t.public_url}")
    if not tunnels:
        print("   Tidak ada tunnel aktif")
except Exception as e:
    print(f"   (skip: {e})")

# ── Jika masih error, kill proses ngrok sepenuhnya ─────────────
try:
    ngrok.kill()
    time.sleep(2)
    print("   ngrok process killed & restarted")
except Exception:
    pass

# ── Buka tunnel baru ───────────────────────────────────────────
time.sleep(1)
tunnel = ngrok.connect(COLAB_PORT)
COLAB_PUBLIC_URL = tunnel.public_url
print(f"\n🌐 Colab URL: {COLAB_PUBLIC_URL}")

# ── Register ke backend ────────────────────────────────────────
print(f"\n📡 Mendaftarkan ke backend: {BACKEND_URL}")
result = backend_post(
    "/colab/register",
    json_data={
        "url": COLAB_PUBLIC_URL,
        "session_id": SESSION_ID,
    },
)

if result and result.get("success"):
    print(f"✅ Berhasil terdaftar!")
    print(f"   URL    : {COLAB_PUBLIC_URL}")
    print(f"   Session: {SESSION_ID}")
else:
    print("❌ Gagal mendaftar ke backend")
    print("   Pastikan BACKEND_URL dan COLAB_API_KEY sudah benar")

Cell 6 — Heartbeat Loop (jalankan dan biarkan berjalan)

In [ ]:
# ── Heartbeat loop ─────────────────────────────────────────────
# Cell ini mengirim ping ke backend setiap 60 detik agar tidak
# dianggap offline. Biarkan cell ini berjalan selama Colab aktif.
# Tekan ■ Stop untuk menghentikan.

HEARTBEAT_INTERVAL = 60  # detik
CONSECUTIVE_FAILS = 0
MAX_FAILS = 5

print("💓 Heartbeat dimulai")
print(f"   Interval : {HEARTBEAT_INTERVAL}s")
print(f"   Session  : {SESSION_ID}")
print(f"   URL      : {COLAB_PUBLIC_URL}")
print("\n   Tekan ■ Stop untuk menghentikan\n")

while True:
    try:
        result = backend_post(
            "/colab/ping",
            json_data={"session_id": SESSION_ID},
        )
        if result:
            CONSECUTIVE_FAILS = 0
            print(
                f"💓 Ping OK [{time.strftime('%H:%M:%S')}]",
                end="\r",
            )
        else:
            CONSECUTIVE_FAILS += 1
            print(f"\n⚠️  Ping gagal ({CONSECUTIVE_FAILS}/{MAX_FAILS})")
            if CONSECUTIVE_FAILS >= MAX_FAILS:
                print("❌ Backend tidak merespons. Coba re-register.")
                # Coba register ulang
                backend_post(
                    "/colab/register",
                    json_data={"url": COLAB_PUBLIC_URL, "session_id": SESSION_ID},
                )
                CONSECUTIVE_FAILS = 0

    except KeyboardInterrupt:
        print("\n\n⏹️  Heartbeat dihentikan")
        backend_post("/colab/unregister", json_data={"session_id": SESSION_ID})
        ngrok.disconnect(COLAB_PUBLIC_URL)
        print("   Colab berhasil unregister dari backend")
        break
    except Exception as e:
        print(f"\n⚠️  Heartbeat error: {e}")

    time.sleep(HEARTBEAT_INTERVAL)